In [1]:
!pip install earthengine-api -q

# BƯỚC 1: IMPORT THƯ VIỆN

In [2]:
import ee
import time
from datetime import datetime

# BƯỚC 2: CÁC THỰC TÀI KHOẢN GEE

In [3]:
print("\n🔐 Đang xác thực Earth Engine...")
print("⚠️  Vui lòng làm theo hướng dẫn để đăng nhập vào tài khoản Google của bạn")

try:
    ee.Authenticate()
    ee.Initialize(project='crafty-booth-478117-i6')  # Thay 'ee-your-project-id' bằng project ID của bạn
    print("✅ Đã khởi tạo Earth Engine thành công!")
except Exception as e:
    print(f"❌ Lỗi khi khởi tạo: {e}")
    raise


🔐 Đang xác thực Earth Engine...
⚠️  Vui lòng làm theo hướng dẫn để đăng nhập vào tài khoản Google của bạn
✅ Đã khởi tạo Earth Engine thành công!


# BƯỚC 3: 🗺️ BIÊN GIỚI VIỆT NAM

In [4]:
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
vietnam = countries.filter(ee.Filter.eq('ADM0_NAME', 'Viet Nam')).first()
vietnamGeometry = vietnam.geometry()

# BƯỚC 4: HÀM XỬ LÝ

In [6]:
def process_aod(aod_band, aod_qa_band, new_name):
    """
    Các bit 0-3 của AOD_QA: Mặt nạ Mây QA (QA Cloud Mask)
    0000 = Chất lượng tốt nhất, 0001 = Chất lượng tốt
    """
    cloud_mask_qa = aod_qa_band.bitwiseAnd(0x000F)
    good_quality_mask = cloud_mask_qa.lte(1) # Giữ lại pixel có chất lượng <= 1
    aod_processed = (
        aod_band.updateMask(good_quality_mask)
        .multiply(0.001)
        .rename(new_name)
    )
    return aod_processed

# BƯỚC 5: 📅 DANH SÁCH NGÀY CHO TỪNG NĂM

In [7]:
dates_by_year = {
 # Năm 2020 - ĐÃ CÓ SẴN
    2020: [
        '2020-01-08','2020-01-16','2020-01-18','2020-01-30',
        '2020-02-12','2020-02-19','2020-02-22','2020-02-24',
        '2020-02-26','2020-03-04','2020-03-10','2020-03-22',
        '2020-03-26','2020-03-29','2020-03-30','2020-04-03',
        '2020-04-05','2020-04-12','2020-04-18','2020-04-21',
        '2020-04-28','2020-04-30','2020-05-04','2020-05-05',
        '2020-05-07','2020-06-01','2020-06-05','2020-06-12',
        '2020-06-20','2020-07-05','2020-07-17','2020-07-26',
        '2020-07-29','2020-08-04','2020-08-18','2020-08-20',
        '2020-08-24','2020-08-25','2020-08-26','2020-08-27',
        '2020-08-30','2020-09-05','2020-09-22','2020-09-30',
        '2020-10-20','2020-10-30','2020-11-07','2020-11-08',
        '2020-11-23','2020-11-26','2020-11-27','2020-11-30',
        '2020-12-06','2020-12-12','2020-12-25','2020-12-29'
    ],

    # Năm 2021 - THÊM NGÀY VÀO ĐÂY
    2021: [
        '2021-01-02','2021-01-16','2021-01-18','2021-01-26',
        '2021-01-28','2021-02-05','2021-02-10','2021-02-12',
        '2021-02-17','2021-02-26','2021-03-01','2021-03-05',
        '2021-03-06','2021-03-07','2021-03-14','2021-03-16',
        '2021-03-22','2021-03-30','2021-04-19','2021-04-24',
        '2021-04-26','2021-05-04','2021-05-10','2021-05-26',
        '2021-05-29','2021-06-17','2021-06-18','2021-06-20',
        '2021-06-21','2021-06-24','2021-07-02','2021-07-03',
        '2021-07-10','2021-07-29','2021-08-16','2021-08-20',
        '2021-08-23','2021-09-28','2021-09-29','2021-10-01',
        '2021-10-02','2021-10-23','2021-10-27','2021-11-03',
        '2021-11-04','2021-11-20','2021-11-22','2021-11-27',
        '2021-12-04','2021-12-05','2021-12-06','2021-12-13',
        '2021-12-18','2021-12-20','2021-12-22','2021-12-25',
        '2021-12-27','2021-12-28'
    ],

    # Năm 2022 - THÊM NGÀY VÀO ĐÂY
    2022: [
        '2022-01-01','2022-01-05','2022-01-06','2022-01-12',
        '2022-01-21','2022-01-22','2022-01-23','2022-01-28',
        '2022-02-02','2022-02-04','2022-02-16','2022-02-22',
        '2022-02-24','2022-02-25','2022-02-28','2022-03-01',
        '2022-03-03','2022-03-04','2022-03-08','2022-03-24',
        '2022-03-26','2022-04-03','2022-04-04','2022-04-07',
        '2022-04-09','2022-04-23','2022-04-27','2022-05-04',
        '2022-05-05','2022-05-20','2022-05-29','2022-06-05',
        '2022-06-24','2022-07-23','2022-07-25','2022-08-16',
        '2022-09-02','2022-09-04','2022-09-11','2022-10-03',
        '2022-10-11','2022-10-13','2022-10-16','2022-10-24',
        '2022-11-04','2022-11-05','2022-11-12','2022-11-27',
        '2022-11-28','2022-11-30','2022-12-03','2022-12-16',
        '2022-12-20','2022-12-24'
    ],

    # Năm 2023 - THÊM NGÀY VÀO ĐÂY
    2023: [
        '2023-01-01','2023-01-03','2023-01-08','2023-01-15',
        '2023-01-26','2023-01-30','2023-01-31','2023-02-02',
        '2023-02-09','2023-02-11','2023-02-22','2023-02-23',
        '2023-02-25','2023-02-26','2023-02-27','2023-03-06',
        '2023-03-09','2023-03-22','2023-03-26','2023-04-05',
        '2023-04-07','2023-04-18','2023-04-23','2023-05-02',
        '2023-05-03','2023-05-09','2023-05-16','2023-05-26',
        '2023-05-29','2023-05-30','2023-06-01','2023-06-03',
        '2023-06-16','2023-06-20','2023-06-21','2023-06-25',
        '2023-07-05','2023-07-06','2023-07-26','2023-08-10',
        '2023-08-11','2023-08-13','2023-08-21','2023-09-20',
        '2023-09-21','2023-09-24','2023-10-09','2023-11-01',
        '2023-11-15','2023-12-26','2023-12-28'
    ],
    # Nam 2024 - them vao day
    2024: [
        '2024-01-02','2024-01-27','2024-02-03','2024-02-05',
        '2024-02-10','2024-02-28','2024-03-08','2024-03-22',
        '2024-04-09','2024-04-16','2024-04-23','2024-05-25',
        '2024-06-12','2024-07-05','2024-08-06','2024-08-09',
        '2024-08-15','2024-09-16','2024-10-05','2024-10-06',
        '2024-10-13','2024-10-14','2024-10-18','2024-10-21',
        '2024-11-10','2024-11-17','2024-11-19','2024-11-23',
        '2024-12-01','2024-12-05','2024-12-21'
    ],
}

# BƯỚC 6: 🧭 CHỌN NĂM CẦN XỬ LÝ

In [17]:
YEAR = 2024 # ← Đổi thành 2020–2024
FOLDER_BASE_NAME = "modis_aod_vietnam"

dates_to_process = dates_by_year[YEAR]
print(f"🔹 Số ngày cần xử lý cho năm {YEAR}: {len(dates_to_process)}")

🔹 Số ngày cần xử lý cho năm 2024: 31


# BƯỚC 7: 🚀 XỬ LÝ MCD19A2 (Optical Depth)

In [18]:
mcd19a2 = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES')

for date_string in dates_to_process:
    print(f"\n🔸 Đang xử lý ngày {date_string}...")

    start_date = f"{date_string}T00:00:00"
    end_date = f"{date_string}T23:59:59"

    filtered = mcd19a2.filterDate(start_date, end_date).filterBounds(vietnamGeometry)
    count = filtered.size().getInfo()
    print(f"   ➜ Số ảnh tìm thấy: {count}")

    if count == 0:
        print(f"⚠️ Không tìm thấy dữ liệu cho ngày {date_string}. Bỏ qua.")
        continue

    # 1️⃣ Tạo mosaic và cắt biên
    image = filtered.mosaic().clip(vietnamGeometry)

    # 2️⃣ Chọn và xử lý các band
    aod_047_raw = image.select('Optical_Depth_047')
    aod_055_raw = image.select('Optical_Depth_055')
    aod_qa = image.select('AOD_QA')

    aod_047 = process_aod(aod_047_raw, aod_qa, 'AOD_047um')
    aod_055 = process_aod(aod_055_raw, aod_qa, 'AOD_055um')

    final_image = ee.Image.cat([aod_047, aod_055]).toFloat()

    # 3️⃣ Xuất ảnh về Google Drive
    output_folder = f"{FOLDER_BASE_NAME}_{YEAR}"
    output_name = f"MCD19A2_Vietnam_AOD_{date_string}"

    task = ee.batch.Export.image.toDrive(
        image=final_image,
        description=output_name,
        folder=output_folder,
        fileNamePrefix=output_name,
        region=vietnamGeometry.bounds(),
        scale=1000,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"✅ Đã tạo tác vụ xuất: {output_name}")

print("\n🎯 Hoàn tất việc tạo tác vụ. Kiểm tra tab 'Tasks' trong Earth Engine Dashboard.")
print(f'🔗 Theo dõi: https://code.earthengine.google.com/tasks')


🔸 Đang xử lý ngày 2024-01-02...
   ➜ Số ảnh tìm thấy: 113
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-01-02

🔸 Đang xử lý ngày 2024-01-27...
   ➜ Số ảnh tìm thấy: 111
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-01-27

🔸 Đang xử lý ngày 2024-02-03...
   ➜ Số ảnh tìm thấy: 89
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-02-03

🔸 Đang xử lý ngày 2024-02-05...
   ➜ Số ảnh tìm thấy: 113
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-02-05

🔸 Đang xử lý ngày 2024-02-10...
   ➜ Số ảnh tìm thấy: 111
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-02-10

🔸 Đang xử lý ngày 2024-02-28...
   ➜ Số ảnh tìm thấy: 97
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-02-28

🔸 Đang xử lý ngày 2024-03-08...
   ➜ Số ảnh tìm thấy: 82
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-03-08

🔸 Đang xử lý ngày 2024-03-22...
   ➜ Số ảnh tìm thấy: 49
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-03-22

🔸 Đang xử lý ngày 2024-04-09...
   ➜ Số ảnh tìm thấy: 83
✅ Đã tạo tác vụ xuất: MCD19A2_Vietnam_AOD_2024-04-